# Medical Sound Classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchaudio
import librosa
import librosa.display
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt

In [2]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

df_train = df_train.drop(df_train[df_train['candidateID'] == '5ee582f2832c2'].index)

df_train['coldPresent_missing'] = df_train['coldPresent'].isna().astype(int)
df_train['coldPresent'] = df_train['coldPresent'].fillna(0)

print(f"train samples: {len(df_train)}")
print(df_train['disease'].value_counts())

train samples: 545
disease
1    238
2    167
0    140
Name: count, dtype: int64


## 2.2 Spectrogrammen genereren

Voor zowel de test & trainings sets gaan we voor elk ID de vowel_processed & cough_processed omzetten naar een spectrogram die we later voor ons CNN gaan gebruiken

In [3]:
def audio_to_melspectrogram(wav_path, save_path, sr=16000, n_mels=128, fmax=8000):
    if os.path.exists(save_path):
        return

    try:
        y, file_sr = librosa.load(wav_path, sr=None)

        if file_sr != sr:
            y = librosa.resample(y, orig_sr=file_sr, target_sr=sr)

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        plt.figure(figsize=(2.24, 2.24), dpi=100)
        plt.axis('off')
        librosa.display.specshow(mel_db, sr=sr, x_axis=None, y_axis=None, cmap='magma')
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
        plt.close()

    except Exception as e:
        print(f"Fout bij {wav_path}: {e}")


def generate_spectrograms(df, split='train'):
    for _, row in df.iterrows():
        cid = row['candidateID']

        os.makedirs(f'spectrograms/{split}/{cid}', exist_ok=True)

        for sound_type in ['cough', 'vowel']:
            wav_path  = f'data/sounds/sounds/{cid}/{sound_type}_processed.wav'
            save_path = f'spectrograms/{split}/{cid}/{sound_type}.png'

            if not os.path.exists(wav_path):
                print(f"Ontbreekt: {wav_path}")
                continue

            audio_to_melspectrogram(wav_path, save_path)

    print(f"✅ Spectrogrammen klaar voor {split} set ({len(df)} patiënten)")


generate_spectrograms(df_train, split='train')
generate_spectrograms(df_test,  split='test')

NameError: name 'os' is not defined

In [4]:
import os
import shutil

def reorganize_by_disease(df, split='train'):
    if 'disease' not in df.columns:
        print(f"Skipping {split} set: no 'disease' column found")
        return

    for _, row in df.iterrows():
        cid = str(row['candidateID'])
        disease = str(int(row['disease'])) 
        
        target_dir = f'spectrograms_final/{split}/{disease}/{cid}'
        os.makedirs(target_dir, exist_ok=True)
        
        for file_name in ['cough.png', 'vowel.png']:
            source_path = f'spectrograms/{split}/{cid}/{file_name}'
            
            if os.path.exists(source_path):
                shutil.copy(source_path, os.path.join(target_dir, file_name))

reorganize_by_disease(df_train, split='train')
print("✅ Spectrograms reorganized by Disease -> Candidate ID")

✅ Spectrograms reorganized by Disease -> Candidate ID


# 4. Train test split

In [5]:
from sklearn.model_selection import train_test_split

X = df_train.drop(columns=['disease', 'candidateID'])
y = df_train['disease']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1234, stratify=y)

print(f"Train: {len(X_train)} samples")
print(f"Val:   {len(X_val)} samples")

Train: 436 samples
Val:   109 samples


In [6]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

(436, 10)
(436,)
(109, 10)
(109,)


In [7]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers

input_shape = (224, 224, 3) 

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
base_model.trainable = False

model = tf.keras.Sequential([
    layers.Input(shape=input_shape),
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(3, activation='softmax') 
])

In [8]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,926 (16.07 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    'spectrograms_final/train',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'spectrograms_final/train',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True
)

# DIT ZOU NU MOETEN WERKEN ZONDER FOUTMELDING
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

Found 1082 files belonging to 3 classes.
Using 866 files for training.
Found 1082 files belonging to 3 classes.
Using 216 files for validation.
Epoch 1/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 26s 510ms/step - accuracy: 0.5173 - loss: 0.9823 - val_accuracy: 0.5231 - val_loss: 1.0074
Epoch 2/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 461ms/step - accuracy: 0.5635 - loss: 0.9458 - val_accuracy: 0.5139 - val_loss: 1.0061
Epoch 3/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 451ms/step - accuracy: 0.5600 - loss: 0.9344 - val_accuracy: 0.5231 - val_loss: 1.0331
Epoch 4/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 465ms/step - accuracy: 0.5416 - loss: 0.9520 - val_accuracy: 0.5093 - val_loss: 1.0206
Epoch 5/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 453ms/step - accuracy: 0.5924 - loss: 0.9105 - val_accuracy: 0.5046 - val_loss: 1.0220
Epoch 6/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 462ms/step - accuracy: 0.5751 - loss: 0.9203 - val_accuracy: 0.3843 - val_loss: 1.0587
Epoch 7/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 460ms/step - accuracy: 0.5589 - loss: 0.

In [12]:
val_loss, val_acc = model.evaluate(val_ds)

print(f"Val loss: {val_loss:.4f}, Val accuracy: {val_acc:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 340ms/step - accuracy: 0.5231 - loss: 1.0074
Val loss: 1.0074, Val accuracy: 0.5231
